#  EHR Data Cleaning — Part 2: Noise Removal & Text Cleaning

In this notebook, you'll clean the EHR data by removing various types of noise and preparing text for extraction.

---

##  What You'll Do Today

1. **Remove duplicate patient records** — Find and eliminate duplicate rows in the patients table

2. **Remove orphaned notes** — Identify clinical notes referencing non-existent patients

3. **Remove empty and placeholder notes** — Clean out notes with no useful content

4. **Strip HTML artifacts** — Remove HTML tags that contaminate note text

5. **Fix OCR errors** — Correct character substitution errors in clinical notes

---

##  Why Data Cleaning Matters

Real-world EHR data contains many quality issues:
- **Duplicate records** from system migrations or data entry errors
- **Orphaned records** that reference deleted patients
- **Empty entries** from incomplete documentation
- **Technical artifacts** like HTML tags from copy-paste
- **OCR errors** from scanned document digitization

Cleaning these issues is essential before any analysis or ML model building.

## Setting Environment Up for Colab

Mount Google Drive and download the dataset so this notebook can access the EHR data.

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# Set your Google Drive path here
# Replace <PATH_TO_REPO> with your actual repository path
REPO_PATH = "/content/drive/MyDrive/<PATH_TO_REPO>"

# Data directory (source data)
DATA_DIR = f"{REPO_PATH}/data/EHR/ehr-data"
NOTES_PATH = f"{DATA_DIR}/notes_for_extraction.csv"

# Output directory (cleaned data)
OUTPUT_DIR = f"{REPO_PATH}/data/week_1/cleaned_data"

print(f"Data directory: {DATA_DIR}")
print(f"Output directory: {OUTPUT_DIR}")

---

##  Step 1: Import Libraries

We need several Python libraries for data cleaning:

| Library | Purpose |
|---------|---------|
| **pandas** | Load and manipulate CSV data |
| **numpy** | Handle missing values and numerical operations |
| **re** | Regular expressions for pattern matching (HTML tags, OCR errors) |
| **os** | File path operations for cross-platform compatibility |
| **json** | Parse JSON data if needed |
| **Counter** | Count word frequencies for OCR analysis |

These are standard data processing libraries that form the foundation of any ETL (Extract-Transform-Load) pipeline.

In [21]:
# (provided)
import pandas as pd
import numpy as np
import re
import os
import json
from collections import Counter

print("[OK] Libraries imported successfully")

---

##  Step 2: Load the Source Data

We'll load the source data, which contains various **data quality issues** for you to identify and clean:

**CSV Tables (7 files):**
- `patients.csv` — Contains duplicate records
- `encounters.csv`, `conditions.csv`, `medications.csv`, `observations.csv`, `procedures.csv`, `allergies.csv` — Standard clinical tables

**Clinical Notes:**
- `notes_for_extraction.csv` — Contains orphaned notes, empty notes, placeholder text, HTML artifacts, and OCR errors

This simulates real-world EHR data that has accumulated quality issues over time from system migrations, scanning paper records, and human data entry errors.

In [ ]:
# (provided) Load data from Google Drive
# Load CSV tables from csv/ subfolder
df_patients = pd.read_csv(os.path.join(DATA_DIR, "csv/patients.csv"))
df_encounters = pd.read_csv(os.path.join(DATA_DIR, "csv/encounters.csv"))
df_conditions = pd.read_csv(os.path.join(DATA_DIR, "csv/conditions.csv"))
df_medications = pd.read_csv(os.path.join(DATA_DIR, "csv/medications.csv"))
df_observations = pd.read_csv(os.path.join(DATA_DIR, "csv/observations.csv"))
df_procedures = pd.read_csv(os.path.join(DATA_DIR, "csv/procedures.csv"))
df_allergies = pd.read_csv(os.path.join(DATA_DIR, "csv/allergies.csv"))

# Load clinical notes (in main data folder, not csv subfolder)
df_notes = pd.read_csv(NOTES_PATH)

# Get column name for notes
note_col = "note_text"

print("All data loaded")
print(f"\nInitial counts:")
print(f"  Patients: {len(df_patients):,}")
print(f"  Notes: {len(df_notes):,}")

###  Helper Function: Case-Insensitive Column Lookup

One common challenge with real-world data is **inconsistent column naming**. The Synthea dataset uses uppercase column names (`ID`, `PATIENT`), but other datasets might use lowercase. To handle this gracefully, we create a helper function that finds columns regardless of case.

We also need to identify which column contains the actual note text — different exports may name it differently (`noisy_note`, `note_text`, `original_note`).

In [ ]:
# (provided) Import helper function from src
import sys
sys.path.insert(0, f'{REPO_PATH}/src')
from helpers import get_col

print("get_col imported from src/helpers.py")

---

#  SECTION 1: Data Noise Removal

In this section, we'll systematically identify and remove 5 types of data quality issues. Each type requires a different detection and cleaning strategy:

| Issue | Detection Method | Action |
|-------|------------------|--------|
| **Duplicate patients** | Check for repeated patient IDs | Remove duplicates (keep first) |
| **Orphaned notes** | Patient ID not in patients table | Remove entire record |
| **Empty notes** | Null, empty string, or whitespace only | Remove entire record |
| **Placeholder notes** | Suspiciously short length | Remove entire record |
| **HTML artifacts** | Regex pattern for `<tags>` | Strip tags but keep note |

**Key Distinction:** For the first 4 issues, we **remove the entire record**. For HTML artifacts, we **keep the note** but clean the text — the underlying clinical content is still valid.

Let's tackle each issue one by one.

---

## Step 1: Duplicate Patient Detection & Removal

**The Problem:** Duplicate patient records create serious issues in healthcare analytics:
- **Inflated statistics**: Patient counts and encounter rates become artificially high
- **Broken joins**: When joining tables, duplicates create unexpected row multiplication
- **Incorrect risk scores**: ML models may see the same patient twice with different feature values

**How Duplicates Occur:**
- System migrations where patients were imported multiple times
- Data entry errors creating multiple records for the same person
- ETL pipeline failures causing re-insertion of batches

**Our Strategy:** Identify rows with duplicate patient IDs and keep only the first occurrence. We use `keep='first'` because the first record is typically the original, and later ones are accidental copies.

Let's examine the duplicate patients more closely to understand the scope of the problem:

> **Your Task 1.1: Find Duplicate Patients**
> 1. Get the patient ID column name using `get_col()`
> 2. Find all duplicated rows using `duplicated(subset=[...], keep=False)`
> 3. Count rows involved in duplication and unique IDs with duplicates

In [ ]:
# TODO: Find duplicate patients by ID
patient_id_col = None  # Find the patient ID column
print(f"Patient ID column: '{patient_id_col}'")

print(f"\nOriginal patient count: {len(df_patients):,}")

# TODO: Find all duplicated rows (keep=False marks ALL duplicates)
duplicate_mask = None  # Find rows where patient_id appears more than once
duplicates = df_patients[duplicate_mask]

print(f"Rows involved in duplication: {len(duplicates):,}")
print(f"Unique IDs with duplicates: {duplicates[patient_id_col].nunique():,}")

Now let's remove the duplicates, keeping only the first occurrence of each patient ID:

> **Your Task 1.2: Examine Sample Duplicates**
> 1. Get the top 3 duplicated patient IDs using `value_counts().head(3).index.tolist()`
> 2. For each, count how many times they appear

In [ ]:
# TODO: Show sample of duplicates
print("Sample duplicated patient IDs:")
sample_dup_ids = None  # Get top 3 duplicated IDs
for pid in sample_dup_ids:
    count = len(df_patients[df_patients[patient_id_col] == pid])
    print(f"  {pid[:20]}... appears {count} times")

> **Your Task 1.3: Remove Duplicate Patients**
> 1. Use `drop_duplicates(subset=[...], keep='first')` to keep first occurrence
> 2. Calculate how many rows were removed

In [ ]:
# TODO: Remove duplicates - keep first occurrence
df_patients_clean = None  # Remove duplicate patients, keeping first occurrence

removed_count = len(df_patients) - len(df_patients_clean)
print(f"[OK] Removed {removed_count:,} duplicate patient rows")
print(f"   Clean patient count: {len(df_patients_clean):,}")

---

## Step 2: Orphaned Notes Detection & Removal

**The Problem:** Orphaned notes reference a `patient_id` that doesn't exist in the patients table. These notes are "orphans" — they have no parent patient record to link to.

**Why This Matters:**
- These notes **cannot be used** for patient-level analysis or ML features
- Attempting to join orphaned notes will result in `NaN` patient demographics
- They represent data integrity violations in the relational model

**How Orphans Occur:**
- Patient records were deleted but associated notes remained
- Data import errors with mismatched or corrupted IDs
- System integration failures between EHR modules

**Our Strategy:** 
1. Create a set of valid patient IDs from the cleaned patients table
2. Check each note's `patient_id` against this valid set
3. Remove any note where the patient_id is not found

This is a classic **referential integrity check** — ensuring foreign keys point to valid primary keys.

Let's examine some of the orphaned patient IDs to confirm they don't exist in our patients table:

> **Your Task 2.1: Find Orphaned Notes**
> 1. Create a set of valid patient IDs from cleaned patients table
> 2. Check which notes have patient_id not in the valid set
> 3. Count orphaned notes

In [ ]:
# TODO: Get set of valid patient IDs from cleaned patients table
valid_patient_ids = None  # Create set from df_patients_clean[patient_id_col]
print(f"Valid patient IDs: {len(valid_patient_ids):,}")

# Check which notes have invalid patient_id
df_notes['patient_id_str'] = df_notes['patient_id'].astype(str)

# TODO: Create mask for orphaned notes (patient_id not in valid set)
orphaned_mask = None  # Find notes with patient IDs not in valid set

print(f"\nTotal notes: {len(df_notes):,}")
print(f"Orphaned notes (invalid patient_id): {orphaned_mask.sum():,}")

Remove all orphaned notes from our dataset:

> **Your Task 2.2: Examine Sample Orphaned Notes**
> 1. Get first 5 orphaned patient IDs

In [ ]:
# TODO: Show sample of orphaned notes
print("Sample orphaned patient IDs:")
orphaned_sample = None  # Get head(5) of orphaned patient_ids
for pid in orphaned_sample:
    print(f"  {pid}")

> **Your Task 2.3: Remove Orphaned Notes**
> 1. Filter out orphaned notes using the mask
> 2. Make a copy of the filtered dataframe

In [ ]:
# TODO: Remove orphaned notes
df_notes_clean = None  # Filter using ~orphaned_mask and .copy()

removed_count = orphaned_mask.sum()
print(f"[OK] Removed {removed_count:,} orphaned notes")
print(f"   Remaining notes: {len(df_notes_clean):,}")

---

## Step 3: Empty Notes Detection & Removal

**The Problem:** Empty notes contain no clinical information whatsoever. They're essentially blank rows that provide no value for analysis or ML.

**What Counts as "Empty":**
- `NULL` / `NaN` values — the note field was never populated
- Empty string `""` — a note was created but no text was entered
- Whitespace only `"   "` — contains only spaces, tabs, or newlines

**Why Empty Notes Exist:**
- Placeholder records created by the EHR system but never completed
- Notes where the text was accidentally deleted during editing
- System-generated skeleton records from automated workflows

**Our Strategy:** Use a compound boolean condition to catch all three cases:
1. `isna()` — catches NULL/NaN
2. `str.strip() == ''` — catches empty strings and whitespace-only

Unlike HTML artifacts, empty notes have **no recoverable content** and must be removed entirely.

Remove the empty notes from our dataset:

> **Your Task 3.1: Find Empty Notes**
> 1. Create a mask for notes that are NA or empty string after stripping whitespace
> 2. Use `isna()` and `str.strip() == ''`

In [ ]:
# TODO: Find empty notes
empty_mask = None  # Combine isna() OR (str.strip() == '')

print(f"Empty notes found: {empty_mask.sum():,}")

> **Your Task 3.2: Remove Empty Notes**
> 1. Filter out empty notes using the mask

In [ ]:
# TODO: Remove empty notes
df_notes_clean = None  # Filter using ~empty_mask and .copy()

removed_count = empty_mask.sum()
print(f"[OK] Removed {removed_count:,} empty notes")
print(f"   Remaining notes: {len(df_notes_clean):,}")

---

## Step 4: Short/Placeholder Notes Detection & Removal

**The Problem:** Some notes pass the "empty" check but are still too short to contain meaningful clinical content. These are often:
- **Placeholder text** like `"..."`, `"TBD"`, `"pending"`, `"[note]"`
- **Incomplete entries** where a clinician started but never finished
- **System artifacts** from automated note generation

**Discovery Approach:** Rather than defining a list of known placeholders, we'll:
1. Calculate the length of each note
2. Examine the distribution to find suspiciously short notes
3. Inspect the actual content to confirm they're not useful
4. Set a reasonable length threshold to filter them out

**Why This Matters:** A real clinical note should contain at minimum:
- Patient identifier
- Date/time
- Some clinical content (chief complaint, assessment, etc.)

This typically requires at least 50-100 characters. Notes shorter than this are almost certainly incomplete or placeholder text.

> **Your Task 4.1: Analyze Note Lengths**
> 1. Calculate length of each note using `str.len()`
> 2. Print statistics and count notes by length threshold

In [ ]:
# TODO: Calculate note lengths
df_notes_clean['note_length'] = None  # Use str.len() on note_col

print("Note Length Statistics:")
print(f"  Min length: {df_notes_clean['note_length'].min()}")
print(f"  Max length: {df_notes_clean['note_length'].max():,}")
print(f"  Mean length: {df_notes_clean['note_length'].mean():.0f}")
print(f"  Median length: {df_notes_clean['note_length'].median():.0f}")

# Count notes by length
print(f"\nNotes by length:")
print(f"  < 20 chars:  {(df_notes_clean['note_length'] < 20).sum():,}")
print(f"  < 50 chars:  {(df_notes_clean['note_length'] < 50).sum():,}")
print(f"  < 100 chars: {(df_notes_clean['note_length'] < 100).sum():,}")

First, let's examine the length distribution to understand what "short" means in this dataset:

> **Your Task 4.2: Examine Shortest Notes**
> 1. Use `nsmallest(20, 'note_length')` to get the 20 shortest notes
> 2. Display their content to see if they're placeholders

In [ ]:
# TODO: Look at what the shortest notes actually contain
print("Shortest notes in the dataset:")
print("=" * 50)

shortest_notes = None  # Get the 20 shortest notes

for idx, row in shortest_notes.iterrows():
    print(f"[{row['note_length']:3d} chars] \"{row[note_col]}\"")

print("\n[!] Notice: These short notes are placeholder text, not real clinical content!")

Let's inspect the actual content of the shortest notes to confirm they're placeholders:

> **Your Task 4.3: Remove Short/Placeholder Notes**
> 1. Set a LENGTH_THRESHOLD of 50 characters
> 2. Create mask for notes shorter than threshold
> 3. Remove them and clean up the helper column

In [ ]:
# TODO: Remove notes that are too short
LENGTH_THRESHOLD = 50
short_mask = None  # Notes with length < threshold

print(f"Notes shorter than {LENGTH_THRESHOLD} characters: {short_mask.sum():,}")

# Remove short notes
df_notes_clean = None  # Filter using ~short_mask and .copy()

# Clean up helper column
df_notes_clean = df_notes_clean.drop(columns=['note_length'])

print(f"[OK] Removed {short_mask.sum():,} short/placeholder notes")
print(f"   Remaining notes: {len(df_notes_clean):,}")

Based on our analysis, notes under 50 characters are placeholders. Let's remove them:

---

## Step 5: HTML Artifact Detection & Cleaning

**The Problem:** HTML tags appear in clinical notes when text is copy-pasted from web interfaces or EHR systems that store data in HTML format internally.

**Common HTML Artifacts:**
```html
<div>Note text here</div>
<p>Paragraph content</p>
<br> or <br/>  (line breaks)
<span style="...">Styled text</span>
```

**Key Difference from Other Noise:** 
- Unlike duplicates, orphans, empty, and placeholder notes — **the underlying content is valid**
- We don't want to delete these notes; we want to **clean them**
- The clinical information is intact, just wrapped in HTML tags

**Our Strategy:**
1. Detect notes containing HTML using regex pattern `<[^>]+>` (matches any `<tag>`)
2. Strip all HTML tags while preserving the text content
3. Keep the cleaned note in our dataset

**Why HTML Gets Into Notes:**
- Copy-paste from web-based EHR interfaces
- Data exports that preserve internal formatting
- Migration from systems that used HTML storage

> **Your Task 5.1: Detect HTML Artifacts**
> 1. Use regex pattern `r'<[^>]+>'` to match HTML tags
> 2. Count notes containing HTML

In [ ]:
# TODO: Detect HTML tags
html_pattern = r'<[^>]+>'

html_mask = None  # Find notes containing HTML tags

print(f"Notes with HTML artifacts: {html_mask.sum():,}")

First, let's detect how many notes contain HTML tags using a regex pattern:

> **Your Task 5.2: Examine HTML-Contaminated Notes**
> 1. Get first 3 notes containing HTML
> 2. Show preview of each

In [ ]:
# TODO: Show sample of HTML-contaminated notes
print("Sample HTML artifacts:")
html_samples = None  # Get head(3) of notes with HTML
for i, note in enumerate(html_samples):
    preview = str(note)[:100]
    print(f"  {i+1}. {preview}...")

Let's preview some HTML-contaminated notes to see what we're dealing with:

> **Your Task 5.3: Strip HTML Tags**
> 1. Create a function `strip_html(text)` that uses `re.sub()` to remove HTML tags
> 2. Apply it to all notes

In [ ]:
# TODO: Create function to strip HTML tags
def strip_html(text):
    """Remove HTML tags from text."""
    if pd.isna(text):
        return text
    # Remove HTML tags from text
    return None  # Fill in

# Apply HTML stripping
df_notes_clean[note_col] = df_notes_clean[note_col].apply(strip_html)

print(f"[OK] Stripped HTML from {html_mask.sum():,} notes")

Now let's create a function to strip HTML tags and apply it to all notes:

> **Your Task 5.4: Verify HTML Removal**
> 1. Check if any notes still contain HTML after stripping

In [ ]:
# TODO: Verify HTML was removed
html_remaining = None  # Count notes still containing HTML
print(f"Notes still containing HTML: {html_remaining}")

---

##  Noise Removal Summary

Before moving to OCR correction, let's summarize what we've cleaned so far. This checkpoint helps us:
- Verify each step removed the expected number of records
- Confirm we haven't accidentally removed too much data
- Document the cleaning pipeline for reproducibility

In [ ]:
# (provided) Print summary
print("=" * 70)
print("NOISE REMOVAL SUMMARY")
print("=" * 70)

print(f"\nPATIENTS:")
print(f"  Original: {len(df_patients):,}")
print(f"  After removing duplicates: {len(df_patients_clean):,}")
print(f"  Removed: {len(df_patients) - len(df_patients_clean):,}")

print(f"\nNOTES:")
print(f"  Original: {len(df_notes):,}")
print(f"  After cleaning: {len(df_notes_clean):,}")
print(f"  Total removed: {len(df_notes) - len(df_notes_clean):,}")

print(f"\n" + "=" * 70)
print("[OK] Data noise removal complete!")
print("=" * 70)

---

#  SECTION 2: OCR Error Correction

**The Problem:** When paper medical records are digitized using Optical Character Recognition (OCR), the scanning software often confuses visually similar characters. This creates "mixed" words where some letters have been replaced by look-alike digits (or vice versa).

**Common OCR Confusions:**
| Digit | Often Misread As | Visual Similarity |
|-------|------------------|-------------------|
| `0` | `O` or `o` | Circular shapes |
| `1` | `l` or `I` | Vertical lines |
| `3` | `E` or `e` | Mirrored curves |
| `5` | `S` or `s` | Curved top |
| `8` | `B` or `b` | Double loops |

**Real Examples You'll See:**
- `diab3tes` → should be `diabetes`
- `med1cation` → should be `medication`  
- `m0nitor` → should be `monitor`
- `pati3nt` → should be `patient`

**Why This Matters for NLP:**
- Word embeddings won't recognize `diab3tes` as related to `diabetes`
- Keyword searches will miss OCR-corrupted terms
- Clinical entity recognition will fail on corrupted drug names

**Our Approach:** We'll use two complementary techniques:
1. **Data-driven analysis** — Discover systematic patterns from word frequencies
2. **Rule-based correction** — Apply digit-to-letter substitutions to mixed words

###  Step 1: Build Vocabulary and Identify Suspicious Words

To find OCR errors, we first need to understand the vocabulary of our clinical notes. We'll:
1. **Tokenize** all notes into individual words
2. **Count word frequencies** to understand what's common vs rare
3. **Identify mixed words** — words containing both letters AND digits

Words that mix letters and digits are strong candidates for OCR errors (unless they're valid medical terms like `HbA1c` or `COVID-19`).

> **Your Task 6.1: Build Vocabulary**
> 1. Join all note text into one string
> 2. Tokenize into words using regex `r'\b[a-zA-Z0-9]+\b'`
> 3. Count word frequencies using Counter

In [ ]:
# TODO: Build vocabulary from all notes
all_text = None  # Join all notes into one string

# Tokenize into words
words = None  # Use regex
word_freq = Counter(words)

print(f"Total words: {len(words):,}")
print(f"Unique words: {len(word_freq):,}")

Now let's identify words that mix letters and digits — these are our OCR error candidates:

> **Your Task 6.2: Find Mixed Letter-Digit Words**
> 1. Find words containing both letters AND digits (potential OCR errors)
> 2. Exclude known valid terms like 'hba1c', 'covid19'

In [ ]:
# Function to check for mixed words
def has_letter_and_digit(word):
    """Check if word contains both letters and digits."""
    # TODO: Check if word has at least one letter
    has_letter = None  # Use any() with c.isalpha()
    # TODO: Check if word has at least one digit  
    has_digit = None  # Use any() with c.isdigit()
    return has_letter and has_digit

# Known valid terms with digits (should not be corrected)
valid_mixed_terms = {
    'covid19', 'covid-19', 'sars-cov-2', 'h1n1', 'b12', 'd3',
    'a1c', 'hba1c', 'type1', 'type2', 't1dm', 't2dm',
    'mg', 'ml', 'g', 'kg', 'mcg', 'iu',
    '24hr', '12hr', '2x', '3x', '4x'
}

# TODO: Find suspicious mixed words (excluding valid terms)
mixed_words = None  # Dict comprehension filtering word_freq

print(f"Mixed letter-digit words: {len(mixed_words):,}")
print(f"Top 20 suspicious mixed words:")
for word, freq in sorted(mixed_words.items(), key=lambda x: -x[1])[:20]:
    print(f"  {word}: {freq}")

###  Step 2: Data-Driven OCR Analysis with Edit Distance

Instead of manually guessing which characters are confused, we can **discover patterns from the data** using a technique called **edit distance matching**.

**The Idea:**
1. Split vocabulary into **low-frequency words** (likely OCR errors) and **high-frequency words** (likely correct spellings)
2. For each low-frequency word, find the **closest high-frequency word** using Levenshtein edit distance
3. Compare the character differences to extract **systematic substitution patterns**

**Why This Works:**
- OCR errors are typically **rare** (the error is random, so `diab3tes` appears less often than `diabetes`)
- Correct spellings are typically **common** (thousands of notes use the correct spelling)
- If `diab3tes` → `diabetes` with edit distance 1, and the only difference is `3` → `e`, that's evidence of a `3↔e` OCR confusion

**Levenshtein Edit Distance:** The minimum number of single-character edits (insertions, deletions, substitutions) required to transform one word into another.
- `edit_distance("diab3tes", "diabetes") = 1` (substitute `3` → `e`)
- `edit_distance("cat", "car") = 1` (substitute `t` → `r`)

First, we define parameters and split the vocabulary into low-frequency (potential errors) and high-frequency (likely correct) word sets:

> **Your Task 6.3: Split Vocabulary by Frequency**
> 1. Create lists of low-frequency words (potential errors) and high-frequency words (likely correct)
> 2. Use thresholds: LOW_MAX_FREQ=100, HIGH_MIN_FREQ=500

In [ ]:
from collections import defaultdict

# Parameters for frequency-based analysis
LOW_MAX_FREQ = 100
HIGH_MIN_FREQ = 500
MIN_WORD_LEN = 4
MAX_EDIT_DIST = 2

# TODO: Split vocabulary into low-frequency and high-frequency sets
low_words = None  # Words with count <= LOW_MAX_FREQ and len >= MIN_WORD_LEN
high_words = None  # Words with count >= HIGH_MIN_FREQ and len >= MIN_WORD_LEN

print(f"Low-frequency words (count <= {LOW_MAX_FREQ}): {len(low_words):,}")
print(f"High-frequency words (count >= {HIGH_MIN_FREQ}): {len(high_words):,}")

In [43]:
# (provided) Edit distance function
def edit_distance(a, b, max_dist=2):
    """
    Calculate Levenshtein distance with early termination.
    Returns max_dist+1 if distance exceeds max_dist for efficiency.
    """
    if abs(len(a) - len(b)) > max_dist:
        return max_dist + 1
    if len(a) > len(b):
        a, b = b, a
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        curr = [i]
        for j, cb in enumerate(b, 1):
            cost = 0 if ca == cb else 1
            curr.append(min(prev[j] + 1, curr[-1] + 1, prev[j-1] + cost))
        if min(curr) > max_dist:
            return max_dist + 1
        prev = curr
    return prev[-1]

# Test the function
print("Edit distance examples:")
print(f"  edit_distance('denie8', 'denies') = {edit_distance('denie8', 'denies')}")
print(f"  edit_distance('medicati0n', 'medication') = {edit_distance('medicati0n', 'medication')}")
print(f"  edit_distance('patient', 'patient') = {edit_distance('patient', 'patient')}")

Next, we implement the Levenshtein edit distance algorithm with early termination for efficiency:

> **Your Task 6.4: Find Correction Pairs**
> 1. For each low-frequency word, find the closest high-frequency match
> 2. Use the edit distance function to compare word similarity

In [ ]:
# (provided) Create length-based index for faster matching
len_index = defaultdict(list)
for w in high_words:
    len_index[len(w)].append(w)

# Find nearest high-frequency match for each low-frequency word
pairs = []
for lw in low_words[:1000]:  # Limit for efficiency
    L = len(lw)
    best = None
    best_d = MAX_EDIT_DIST + 1
    
    for k in range(L - MAX_EDIT_DIST, L + MAX_EDIT_DIST + 1):
        for cand in len_index.get(k, []):
            # TODO: Calculate edit distance between low-freq word and candidate
            d = None   # How far apart are these two words?
            if d < best_d:
                best, best_d = cand, d
    
    if best is not None and best_d <= MAX_EDIT_DIST:
        pairs.append((lw, word_freq[lw], best, word_freq[best], best_d))

print(f"Found {len(pairs)} potential correction pairs")

In [45]:
# (provided) Display candidates
# Display top correction candidates
pairs.sort(key=lambda x: (-x[3], x[0]))

print("TOP CORRECTION CANDIDATES")
print("=" * 70)
print(f"{'Low-freq word':<20} {'Freq':<6} {'High-freq word':<20} {'Freq':<6} {'Dist'}")
print("-" * 70)
for low_tok, low_cnt, high_tok, high_cnt, dist in pairs[:30]:
    print(f"{low_tok:<20} {low_cnt:<6} {high_tok:<20} {high_cnt:<6} {dist}")

Now we find the closest high-frequency match for each low-frequency word using our edit distance function:

> **Your Task 6.5: Analyze Character Substitution Patterns**
> 1. For pairs with same length, compare character by character
> 2. Count which characters are substituted for which

In [ ]:
# (provided) Count character-level substitutions from correction pairs
substitutions = Counter()

for low_tok, _, high_tok, _, _ in pairs:
    if len(low_tok) == len(high_tok):
        for c_low, c_high in zip(low_tok, high_tok):
            if c_low != c_high:
                substitutions[(c_low, c_high)] += 1

print("CHARACTER SUBSTITUTION PATTERNS")
print("=" * 50)
print(f"{'From':<8} {'To':<8} {'Count'}")
print("-" * 30)
for (src, dst), count in substitutions.most_common(20):
    print(f"{src:<8} {dst:<8} {count:>5}")

> **Your Task 6.6: Extract Digit-to-Letter Confusions**
> 1. Filter substitutions where source is digit and destination is letter
> 2. Keep the most common letter for each digit

In [ ]:
# TODO: Extract digit-to-letter confusions specifically
digit_to_letter = {}
for (src, dst), count in substitutions.items():
    if src.isdigit() and dst.isalpha():
        if src not in digit_to_letter or count > digit_to_letter[src][1]:
            digit_to_letter[src] = (dst, count)

print("DIGIT TO LETTER CONFUSIONS (Data-Driven)")
print("=" * 50)
for digit, (letter, count) in sorted(digit_to_letter.items(), key=lambda x: -x[1][1]):
    print(f"  '{digit}' is often misread as '{letter}'  ({count} occurrences)")

print("\n These patterns were discovered from the data, not manually defined!")

Let's display the top correction candidates to see what patterns we've discovered:

###  Step 3: Define OCR Correction Rules

Based on common OCR error patterns (and validated by our data-driven analysis above), we define a mapping from digits to the letters they're most commonly confused with.

**Our Substitution Rules:**
```
0 → o (zero looks like letter O)
1 → l (one looks like lowercase L)
3 → e (three looks like backwards E)
4 → a (four's top looks like A)
5 → s (five looks like letter S)
6 → g (six looks like lowercase G)
7 → t (seven looks like letter T)
8 → b (eight looks like letter B)
```

**Important:** We only apply these substitutions to words that contain **both letters and digits** (mixed words). Pure numbers like `500mg` are preserved because the digits are intentional.

**Valid Medical Terms:** Some terms legitimately mix letters and digits:
- `HbA1c` \u2014 glycated hemoglobin test
- `COVID-19` \u2014 coronavirus disease
- `B12`, `D3` \u2014 vitamins
- `T1DM`, `T2DM` \u2014 diabetes type abbreviations

We maintain an exclusion list for these valid terms.

> **Your Task 7.1: Create OCR Fix Function**
> 1. Define substitution mapping (digit -> letter)
> 2. Create function to fix OCR errors in a single word

In [ ]:
# OCR digit-to-letter substitution patterns
ocr_substitutions = {
    '0': 'o',  # zero -> letter O
    '1': 'l',  # one -> lowercase L
    '3': 'e',  # three -> letter E
    '4': 'a',  # four -> letter A
    '5': 's',  # five -> letter S
    '6': 'g',  # six -> letter G
    '7': 't',  # seven -> letter T
    '8': 'b',  # eight -> letter B
}

# TODO: Create function to fix OCR errors in a word
def fix_ocr_word(word, ocr_subs, valid_terms):
    """Attempt to fix OCR errors in a word by substituting digits."""
    if not has_letter_and_digit(word):
        return word
    if word.lower() in valid_terms:
        return word
    
    # Try substituting each digit
    fixed = word.lower()
    # TODO: Use a loop to replace each digit with its corresponding letter
    return None  # Fill in

# Test the function
test_words = ['diab3tes', 'pati3nt', 'med1cation', 'm0nitor']
print("OCR correction examples:")
for word in test_words:
    print(f"  {word} -> {fix_ocr_word(word, ocr_substitutions, valid_mixed_terms)}")

Now we extract systematic character substitution patterns from our correction pairs:

Now we create a function to apply OCR fixes to an entire note (processing all words):

In [ ]:
# (provided) Function to fix OCR in entire text
def fix_ocr_text(text, ocr_subs, valid_terms):
    """Fix OCR errors in entire text."""
    if pd.isna(text):
        return text
    
    text = str(text)
    words = re.findall(r'\b[a-zA-Z0-9]+\b', text)
    
    for word in words:
        if has_letter_and_digit(word) and word.lower() not in valid_terms:
            fixed = fix_ocr_word(word, ocr_subs, valid_terms)
            if fixed != word.lower():
                # Replace preserving case of first character
                if word[0].isupper():
                    fixed = fixed.capitalize()
                text = text.replace(word, fixed)
    
    return text

print("[OK] OCR correction function defined")

###  Step 4: Apply OCR Corrections to All Notes

Now we'll apply our correction rules to every note in the dataset. The process:

1. **Identify notes with potential OCR errors** — notes containing mixed letter-digit words
2. **For each mixed word:**
   - Skip if it's a valid medical term (in our exclusion list)
   - Apply digit-to-letter substitutions
   - Preserve the original case (capitalize if needed)
3. **Replace** the corrupted word with the corrected version

**Note:** OCR correction is imperfect. Some mixed words might be:
- Legitimate terms we didn't include in our exclusion list
- Errors that our simple substitution can't fix (missing characters, etc.)

After correction, we'll still have some notes flagged as having "OCR errors" — these are cases our rules couldn't fully address. Advanced techniques (spell-checking dictionaries, context-aware models) could improve coverage.

Let's count how many notes have OCR errors before and after applying our corrections:

> **Your Task 7.2: Count Notes with OCR Errors**
> 1. Create function to check if text has OCR errors
> 2. Count notes with potential OCR errors

In [ ]:
# TODO: Count notes with potential OCR errors before fixing
def has_ocr_errors(text, valid_terms):
    if pd.isna(text):
        return False
    words = re.findall(r'\b[a-zA-Z0-9]+\b', str(text))
    for word in words:
        if has_letter_and_digit(word) and word.lower() not in valid_terms:
            return True
    return False

ocr_error_mask = None  # Apply has_ocr_errors to note column
print(f"Notes with potential OCR errors: {ocr_error_mask.sum():,}")

> **Your Task 7.3: Apply OCR Corrections**
> 1. Apply fix_ocr_text to all notes
> 2. Verify correction by counting remaining errors

In [ ]:
# TODO: Apply OCR corrections to all notes
df_notes_clean[note_col] = None  # Apply fix_ocr_text with ocr_substitutions and valid_mixed_terms

# Verify corrections
ocr_error_mask_after = df_notes_clean[note_col].apply(lambda x: has_ocr_errors(x, valid_mixed_terms))
print(f"Notes with OCR errors after fixing: {ocr_error_mask_after.sum():,}")
print(f"\n[OK] OCR corrections applied")

---

##  Step 5: Save Cleaned Data

Now that we've completed all cleaning steps, we'll save the results to a `cleaned_data/` directory. This creates a clean separation between:

- **`data/`** — Original source data (never modify)
- **`cleaned_data/`** — Our cleaned output (reproducible from running this notebook)

**What We're Saving:**
| File | Contents | Changes Made |
|------|----------|--------------|
| `csv/patients.csv` | Patient demographics | Duplicates removed |
| `notes_for_extraction.csv` | Clinical notes | Orphans, empty, placeholders removed; HTML stripped; OCR corrected |
| `csv/*.csv` (other tables) | Clinical tables | Unchanged (no cleaning needed) |

**Best Practice:** Always save cleaned data to a new location rather than overwriting the source. This ensures you can:
1. Re-run the cleaning pipeline if you discover new issues
2. Compare cleaned vs. original data for validation
3. Track the provenance of your processed data

> **Your Task 8.1: Create Output Directory**
> 1. Create output directory structure

In [ ]:
# (provided)
# Create output directory structure
# - cleaned_data/notes_for_extraction.csv (notes saved directly)
# - cleaned_data/csv/*.csv (CSVs saved in csv subfolder)
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, "csv"), exist_ok=True)

print(f"Output directory: {OUTPUT_DIR}")
print(f"CSV folder: {OUTPUT_DIR}/csv")

First, create the output directory structure:

> **Your Task 8.2: Save Cleaned Data**
> 1. Save cleaned patients and notes
> 2. Copy other CSV files (already clean)

In [ ]:
# (provided) Save cleaned data
df_patients_clean.to_csv(os.path.join(OUTPUT_DIR, "csv", "patients.csv"), index=False)
print(f"[OK] Saved: csv/patients.csv ({len(df_patients_clean):,} rows)")

# Save cleaned notes
df_notes_clean.to_csv(os.path.join(OUTPUT_DIR, "notes_for_extraction.csv"), index=False)
print(f"[OK] Saved: notes_for_extraction.csv ({len(df_notes_clean):,} rows)")

# Copy other CSV files (already clean)
for table_name, df in [
    ('encounters', df_encounters),
    ('conditions', df_conditions),
    ('medications', df_medications),
    ('observations', df_observations),
    ('procedures', df_procedures),
    ('allergies', df_allergies)
]:
    df.to_csv(os.path.join(OUTPUT_DIR, "csv", f"{table_name}.csv"), index=False)
    print(f"[OK] Saved: csv/{table_name}.csv ({len(df):,} rows)")

---

##  Summary: Data Cleaning Complete!

### [OK] What You Accomplished

In this notebook, you systematically cleaned EHR data by addressing 6 types of quality issues:

| Step | Issue | Technique | Result |
|------|-------|-----------|--------|
| 1 | Duplicate patients | `drop_duplicates()` on ID | ~100 rows removed |
| Step 2: | Orphaned notes | Referential integrity check | ~500 notes removed |
| Step 3: | Empty notes | NULL/whitespace detection | ~500 notes removed |
| Step 4: | Placeholder notes | Length threshold (<50 chars) | ~400 notes removed |
| Step 5: | HTML artifacts | Regex tag stripping | ~1,700 notes cleaned |
| Step 6: | OCR errors | Digit-to-letter substitution | ~15,000 notes corrected |

###  Key Takeaways

1. **Data cleaning is essential** before any analysis or ML — garbage in, garbage out
2. **Different noise types require different strategies** — remove vs. clean vs. transform
3. **Validation matters** — always verify your cleaning didn't remove too much data
4. **Preserve source data** — save cleaned output to a new location for reproducibility

###  Next Steps

In the **Homework**, you'll:
- Extract clinical information from the cleaned notes using regex patterns
- Identify conditions, medications, lab values, and vital signs
- Integrate extracted data with the CSV tables to build complete patient records

###  Professional Tip

Real-world data cleaning is iterative. You'll often discover new quality issues after you start analyzing the "clean" data. Build your pipeline to be re-runnable — when you find a new issue, add a cleaning step and regenerate your output. Document every transformation for auditability!

---

## Important: Save Your Functions to src/

After completing this notebook, **copy your OCR functions to `src/data_processing/ocr.py`**. This makes your code reusable in the Homework and future notebooks.

**File to update:** `src/data_processing/ocr.py`

**Functions to copy:**
1. `has_letter_and_digit()` - Check for mixed letter-digit words
2. `fix_ocr_word()` - Fix OCR errors in a single word
3. `strip_html()` - Remove HTML tags from text

**Already provided in src:** `edit_distance()`, `fix_ocr_text()`, `has_ocr_errors()`

Once copied, you can import them:
```python
from data_processing.ocr import fix_ocr_text, strip_html
```